<a href="https://colab.research.google.com/github/nig414/AML-experiments/blob/main/AML_Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif, VarianceThreshold

# 1. Load the dataset (same as Experiment 1)
print("Please upload the 'iris_synthetic_data.csv' file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

In [ ]:
# 2. Preprocess the dataset
# Impute missing values with median
df = df.fillna(df.median(numeric_only=True))

# Encode target variable 'label' to numerical values
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

# Separate features (X) and target (y)
X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

print("\n--- Feature Selection Experiment Started ---")

# Define a function to display top features for each method
def display_top_features(method_name, scores, feature_names):
    feature_scores = pd.Series(scores, index=feature_names)
    top_features = feature_scores.sort_values(ascending=False)
    print(f"\n[Method: {method_name}]")
    print(top_features)
    return top_features

# Method 1: Variance Threshold (Filters low-variance features)
vt = VarianceThreshold()
vt.fit(X)
display_top_features("Variance Threshold", vt.variances_, X.columns)

# Method 2: Pearson Correlation (Feature correlation with target)
corr_scores = X.corrwith(y).abs()
display_top_features("Pearson Correlation (Absolute)", corr_scores.values, X.columns)

# Method 3: ANOVA F-test (Linear dependency between feature and target)
f_selector = SelectKBest(score_func=f_classif, k='all')
f_selector.fit(X, y)
display_top_features("ANOVA F-test", f_selector.scores_, X.columns)

# Method 4: Mutual Information (Information gain - captures any dependency)
mi_scores = mutual_info_classif(X, y, random_state=42)
display_top_features("Mutual Information", mi_scores, X.columns)

# Method 5: Chi-Square Test (Measures association for non-negative features)
chi2_selector = SelectKBest(score_func=chi2, k='all')
chi2_selector.fit(X, y)
display_top_features("Chi-Square Test", chi2_selector.scores_, X.columns)

# Visualizing the scores comparison
plt.figure(figsize=(12, 6))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix for Feature Selection")
plt.show()